## Primera Llamada al Modelo

En este ejercicio, aprenderemos a realizar nuestra primera llamada a un modelo de lenguaje usando la API de Groq

# 1.  Groq API - Conexión Directa con OpenAI Client

## Objetivos de Aprendizaje
- Configurar una conexión directa con Groq usando el cliente OpenAI
- Comprender los parámetros básicos de configuración de API
- Implementar llamadas básicas a modelos de lenguaje
- Aplicar mejores prácticas de seguridad con API keys

## Introducción
Groq proporciona acceso a varios modelos de lenguaje mediante una API compatible con OpenAI. En este notebook aprenderemos a:
1. Configurar el entorno y las credenciales
2. Establecer una conexión con la API
3. Realizar llamadas básicas al modelo
4. Explorar diferentes parámetros de configuración

## Configuración de Variables de Entorno

Antes de ejecutar el código, asegúrate de tener configuradas las siguientes variables de entorno:

```bash
export GROQ_BASE_URL="https://api.groq.com/openai/v1"
export GROQ_API_KEY="tu_token_aqui"
```

**Mejores Prácticas de Seguridad:**
- Nunca hardcodees API keys en el código
- Usa variables de entorno o archivos .env
- No compartas credenciales en repositorios públicos
- Rota las API keys regularmente

## Instalación de Dependencias
```bash
pip install openai python-dotenv
```

- `openai`: cliente que permite conectar con cualquier proveedor compatible con el formato OpenAI (Groq, Mistral, OpenRouter, etc.)
- `python-dotenv`: permite cargar las variables de entorno (API keys, URLs) desde el archivo `.env` con `load_dotenv()`

In [1]:
!pip install openai python-dotenv


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 26.0 MB/s  0:00:00

   ----- ----------------------------------  2/14 [sniffio]
   -------- -------------------------------  3/14 [python-dotenv]
   ----------------- ----------------------  6/14 [idna]
   ----------------- ----------------------  6/14 [idna]
   -------------------- -------------------  7/14 [h11]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [pydantic]
   ------------------------- --------------  9/14 [

In [ ]:
# Importar las bibliotecas necesarias
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

# Configuración del cliente OpenAI para Groq Models

    # Configurar el cliente con variables de entorno
client = OpenAI(
    base_url=os.environ.get("GROQ_BASE_URL"),
    api_key=os.environ.get("GROQ_API_KEY")
)


OpenAI library version: 3.6.0
Python version: 3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]


In [ ]:
# Primera llamada básica al modelo
def llamada_basica():
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "user", "content": "Hola, ¿cómo estás? Responde en una oración."}
            ],
            temperature=0.1,
            max_tokens=1000
        )

        print("=== Respuesta del Modelo ===")
        print(response.choices[0].message.content) # respuesta del modelo
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {response.model}")
        print(f"Tokens usados: {response.usage.total_tokens}")
        print(f"Tokens de entrada: {response.usage.prompt_tokens}")
        print(f"Tokens de salida: {response.usage.completion_tokens}")

    except Exception as e:
        print(f"Error en la llamada: {e}")
        print("Verifica tu configuración y conexión a internet")

# Ejecutar la función
llamada_basica()

Error en la llamada: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}
Verifica tu configuración y conexión a internet


## Usando Roles del Sistema

El rol "system" permite establecer el comportamiento y contexto del asistente antes de la conversación.

In [ ]:
# Ejemplo con mensaje de sistema
def usar_mensaje_sistema():
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {
                    "role": "system",
                    "content": "Eres un experto en tecnología que explica conceptos complejos de manera simple y amigable. Siempre incluyes ejemplos prácticos."
                },
                {
                    "role": "user",
                    "content": "¿Qué es La Inteligencia Artificial?"
                }
            ],
            temperature=0.1,
            max_tokens=1000
        )

        print("=== Respuesta con Mensaje de Sistema ===")
        print(response.choices[0].message.content)

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar función
usar_mensaje_sistema()

Error: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}


## Explorando Parámetros de Configuración

Los parámetros más importantes al hacer llamadas a LLMs son:

- **temperature**: Controla la creatividad (0.0 = determinístico, 1.0 = muy creativo)
- **max_tokens**: Límite de tokens en la respuesta
- **model**: El modelo específico a usar (openai/gpt-oss-120b, openai/gpt-oss-20b, etc.)
- **messages**: Array de mensajes con roles (system, user, assistant)

In [ ]:
# Comparando diferentes valores de temperature
def comparar_temperature():
    prompt = "Escribe una historia muy corta sobre un robot que aprende a cocinar."

    temperatures = [0.1, 0.5, 0.9]

    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)

        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=2000
            )

            print(response.choices[0].message.content)
            print(f"\nTokens usados: {response.usage.total_tokens}")

        except Exception as e:
            print(f"Error: {e}")


# Ejecutar comparación
comparar_temperature()


TEMPERATURE: 0.1
Error: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}

TEMPERATURE: 0.5
Error: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}

TEMPERATURE: 0.9
Error: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}


### Ejercicio 1: Experimentar con Diferentes Modelos
Modifica el código para probar los distintos modelos de Groq:
- openai/gpt-oss-120b
- llama-3.3-70b-versatile
- openai/gpt-oss-20b

Revisa el catálogo vigente en la [documentación de Groq API](https://console.groq.com/docs/models)

**Probar otro proveedor:**
Si quieres, puedes repetir el ejercicio cambiando `base_url`, `api_key` y `model` del cliente para usar otro proveedor:

| Proveedor | Modelos de ejemplo (capa gratuita) |
|---|---|
| Mistral | mistral-small-latest, mistral-large-latest |
| OpenRouter | modelos con sufijo (ej. nvidia/nemotron-3-ultra-550b-a55b:free) |

Documentación: [Mistral](https://docs.mistral.ai/api) · [OpenRouter](https://openrouter.ai/models)

### Ejercicio 2: Crear un Asistente Especializado
Diseña un mensaje de sistema para crear un asistente especializado en un tema específico (ejemplo: finanzas, salud, educación).

### Ejercicio 3: Optimización de Tokens
Experimenta con diferentes valores de max_tokens para encontrar el equilibrio entre respuesta completa y eficiencia de costos.